[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Despliegue en *Cloud Run*.

*Cloud Run* es un servicio de *Google Cloud Platform* (*GCP*) que ejecuta un contenedor de forma gestionada: recibe una imagen, la pone a la escucha de peticiones *HTTP* (*Hypertext Transfer Protocol*) y escala el número de instancias según la demanda.

Este cuaderno hace un **primer despliegue imperativo**. Se usan los guiones de ```api/deploy/gcp/``` y el *workflow* ```.github/workflows/deploy-cloud-run.yml```. La autenticación del *workflow* es *Workload Identity Federation* (*WIF*, federación de identidad de cargas de trabajo) con *OpenID Connect* (*OIDC*): no hay llave de cuenta de servicio. La imagen que se despliega es la que el *CI* ya publicó en *GitHub Container Registry* (*GHCR*), promocionada a *Artifact Registry* con la misma etiqueta *SHA* (*Secure Hash Algorithm*). No se usa *Terraform* ni se aprovisiona una base de datos gestionada.

## Objetivos.

* Identificar los conceptos de proyecto, cuenta de servicio y roles de *IAM* (*Identity and Access Management*), con privilegio mínimo.
* Configurar *WIF* con *OIDC* para este repositorio y el entorno protegido ```cloudrun```.
* Hacer un primer despliegue manual (que concede el acceso público una vez) y repetir el despliegue desde el *workflow*, que promociona la imagen de *GHCR* a *Artifact Registry* y comprueba ```GET /health```.

## Qué entra en este laboratorio y qué no.

**Sí se practica:** usar un repositorio de *Artifact Registry*, promocionar la imagen ya publicada en *GHCR*, desplegar un servicio de *Cloud Run*, autenticar el *workflow* con *WIF* y comprobar su *URL* (*Uniform Resource Locator*) pública.

**Queda fuera de este curso** (se introduce en cursos posteriores):

* *Cloud SQL* (base de datos relacional gestionada).
* *Secret Manager* (almacén de secretos de *GCP*).
* *Compute Engine* (máquinas virtuales).
* *Terraform* e infraestructura como código.

Al cierre hay un mapa breve de esos servicios, solo para saber qué son.

## Proyecto, región, *IAM* y entorno protegido.

Un **proyecto** de *GCP* agrupa recursos, facturación y permisos. La **región** es la zona geográfica en la que se crea el servicio; la elige quien cursa el taller (o quien imparte el curso) y se guarda en la variable ```GCP_REGION``` del entorno ```cloudrun```. No se asume una región concreta en este material.

*IAM* decide quién puede hacer qué. En este laboratorio intervienen, como mínimo:

* El **proyecto**, identificado por ```GCP_PROJECT_ID```.
* El **repositorio de *Artifact Registry***, identificado por ```AR_REPOSITORY```.
* El **servicio de *Cloud Run***, identificado por ```CLOUD_RUN_SERVICE```.
* La **cuenta de servicio de despliegue** (```GCP_DEPLOY_SA```, por defecto ```cd441-deployer```): publica imágenes en *ese* repositorio de *Artifact Registry*, actualiza el servicio de *Cloud Run* y puede «actuar como» solo la cuenta de ejecución. **No** puede cambiar políticas *IAM*.
* La **cuenta de servicio de ejecución** (```GCP_RUNTIME_SA```, por defecto ```cd441-runtime```): es la identidad con la que corre el contenedor, distinta de la que despliega, y no tiene roles (la *API* no usa *GCP*).
* El **proveedor de *WIF*** (```GCP_WIF_PROVIDER```).

Todas esas variables viven en el entorno protegido ```cloudrun``` de *GitHub* (*Settings* → *Environments*), no a nivel de repositorio. No hace falta ningún secreto. Cada estudiante trabaja sobre su propio proyecto de *GCP* y su propio repositorio de *GitHub*. Nada de eso se escribe como un valor fijo en el código.

## Autenticación: la persona en el *Codespace*, el *workflow* con *WIF*.

Hay dos momentos distintos:

1. **Preparación y primer despliegue**, en la terminal del *Codespace* (o en *Cloud Shell*), con la cuenta propietaria del proyecto y el *SDK* de *Google Cloud* (*Software Development Kit*), ya instalado en el contenedor de desarrollo. Quien ejecuta los guiones se autentica como sí mismo en el *SDK* y define, solo en esa sesión, ```GCP_PROJECT_ID``` y ```GCP_REGION```.
2. **En *GitHub Actions***, el *workflow* no usa llaves. *GitHub* emite un testigo *OIDC* de corta duración (el *job* declara ```id-token: write```) y *GCP* lo canjea por un testigo temporal, solo si proviene de **este** repositorio y del entorno ```cloudrun``` (condición de atributos). Un *fork* u otro repositorio no pasa.

Esa es la vía recomendada que el cuaderno 13 presentó y que aquí se practica. No hay un secreto con una llave *JSON* de cuenta de servicio.

## Preparación del proyecto y de *WIF*.

Los guiones están en ```api/deploy/gcp/``` y son idempotentes. ```preparar_proyecto.sh``` habilita las *API* necesarias, crea el repositorio de *Artifact Registry* y las dos cuentas de servicio con privilegio mínimo. **No** crea llaves *JSON*. ```configurar_wif.sh``` crea el fondo y el proveedor *OIDC*, con una condición que exige este repositorio y el entorno ```cloudrun```, y concede a esa identidad el rol ```workloadIdentityUser``` solo sobre la cuenta de despliegue.

Antes de ejecutarlos, hay que definir en la sesión ```GCP_PROJECT_ID``` y ```GCP_REGION``` (sin versionarlas). El segundo guion espera además ```GITHUB_REPO``` con el propietario y el nombre del repositorio, separados por una barra (en un *Codespace* suele coincidir con ```GITHUB_REPOSITORY```).

La siguiente celda preparará el proyecto: *API*, registro y cuentas de servicio de privilegio mínimo.

In [ ]:
bash api/deploy/gcp/preparar_proyecto.sh

La siguiente celda configurará *Workload Identity Federation* para este repositorio y el entorno ```cloudrun```.

In [ ]:
GITHUB_REPO="${GITHUB_REPOSITORY}" bash api/deploy/gcp/configurar_wif.sh

El guion imprime el valor de ```GCP_WIF_PROVIDER``` que hay que cargar, junto con las demás variables, en el entorno ```cloudrun``` del repositorio.

## Primer despliegue manual (acceso público, una vez).

El *workflow* **no** usa ```--allow-unauthenticated```: la cuenta de despliegue no puede cambiar *IAM*. El acceso público se concede **una vez**, a mano, en el primer despliegue, porque lo hace la persona propietaria del proyecto. Ese primer despliegue construye la imagen del *commit* actual, la publica en *Artifact Registry* y deja el servicio accesible. Los siguientes los hace el *workflow*, ya promocionando la imagen de *CI*.

Si la organización aplica la política de restricción de dominio, ```--allow-unauthenticated``` puede fallar: en ese caso se usa un proyecto personal fuera de esa política, o se comprueba el servicio con una identidad autorizada.

La siguiente celda ejecutará el primer despliegue manual y comprobará ```GET /health```.

In [ ]:
bash api/deploy/gcp/desplegar_manual.sh

El guion llama a ```comprobar.sh```, que muestra la *URL* del servicio y hace una petición a ```GET /health```. En *Cloud Run* no hay ```DATABASE_URL```, así que la respuesta típica incluye ```"almacen":"memoria"``` y ```"status":"ok"```, con ```"version"``` igual al *SHA* del *commit*.

## El *workflow* ```deploy-cloud-run.yml```.

El archivo vigente es ```.github/workflows/deploy-cloud-run.yml```. Se dispara de forma manual (*workflow_dispatch*) o al empujar una etiqueta que empiece por ```v```. Usa *runners* hospedados, el entorno protegido ```cloudrun``` y ```concurrency``` sin cancelar un despliegue a medias. Si el disparo fue una etiqueta, además de la etiqueta *SHA* se publica una etiqueta de versión; la que se despliega sigue siendo la del *SHA*.

El siguiente bloque reproduce el contenido de referencia de ese *workflow*.

```yaml
# Despliegue en Cloud Run (cuaderno 16).
#
# Flujo: autenticarse en GCP SIN llaves (OIDC + Workload Identity Federation) →
# tomar la imagen ya publicada en GHCR para este commit → copiarla a Artifact
# Registry con la misma etiqueta (SHA) → `gcloud run deploy` → comprobar /health.
#
# Buenas prácticas que muestra este archivo:
#  - Sin llave JSON de cuenta de servicio: GitHub emite un token OIDC de corta
#    duración y GCP lo canjea, solo si viene de ESTE repositorio y del entorno
#    `cloudrun` (condición de atributos, ver api/deploy/gcp/configurar_wif.sh).
#  - `id-token: write` solo en este job: permite pedir el token, no da acceso al repo.
#  - Entorno protegido `cloudrun`: reglas de rama/etiqueta y, si se quiere,
#    aprobación manual antes de desplegar.
#  - Cuentas de servicio con privilegio mínimo: la que despliega no puede cambiar
#    políticas IAM; el contenedor corre con otra cuenta, sin roles.
#  - Se despliega la imagen probada en CI (misma etiqueta SHA): no se reconstruye.
#  - `concurrency` sin cancelar: nunca dos despliegues a la vez, ni uno a medias.
#
# Variables del entorno `cloudrun` (Settings → Environments → cloudrun). No son secretos:
#   GCP_PROJECT_ID, GCP_REGION, AR_REPOSITORY, CLOUD_RUN_SERVICE,
#   GCP_WIF_PROVIDER, GCP_DEPLOY_SA, GCP_RUNTIME_SA
# No se necesita ningún secreto.
name: Despliegue en Cloud Run

on:
  workflow_dispatch:
  push:
    tags:
      - "v*"

permissions:
  contents: read

concurrency:
  group: deploy-cloud-run
  cancel-in-progress: false

jobs:
  desplegar:
    runs-on: ubuntu-latest
    timeout-minutes: 20
    environment:
      name: cloudrun
      url: ${{ steps.desplegar.outputs.url }}
    permissions:
      contents: read
      id-token: write # pedir el token OIDC a GitHub
    env:
      GCP_PROJECT_ID: ${{ vars.GCP_PROJECT_ID }}
      GCP_REGION: ${{ vars.GCP_REGION }}
      AR_REPOSITORY: ${{ vars.AR_REPOSITORY }}
      CLOUD_RUN_SERVICE: ${{ vars.CLOUD_RUN_SERVICE }}
      GCP_RUNTIME_SA: ${{ vars.GCP_RUNTIME_SA }}
    steps:
      - name: Comprobar y validar variables
        env:
          GCP_WIF_PROVIDER: ${{ vars.GCP_WIF_PROVIDER }}
          GCP_DEPLOY_SA: ${{ vars.GCP_DEPLOY_SA }}
        run: |
          faltan=0
          for nombre in GCP_PROJECT_ID GCP_REGION AR_REPOSITORY CLOUD_RUN_SERVICE \
                        GCP_WIF_PROVIDER GCP_DEPLOY_SA GCP_RUNTIME_SA; do
            if [ -z "${!nombre}" ]; then
              echo "::error::Falta la variable ${nombre} en el entorno cloudrun."
              faltan=1
            fi
          done
          [ "$faltan" -eq 0 ] || exit 1
          # Validación de formato: estos valores acaban dentro de comandos.
          [[ "$GCP_PROJECT_ID" =~ ^[a-z][a-z0-9-]{4,28}[a-z0-9]$ ]] \
            || { echo "::error::GCP_PROJECT_ID no válido"; exit 1; }
          [[ "$GCP_REGION" =~ ^[a-z]+-[a-z]+[0-9]+$ ]] || { echo "::error::GCP_REGION no válida"; exit 1; }
          [[ "$AR_REPOSITORY" =~ ^[a-z][a-z0-9-]{0,62}$ ]] || { echo "::error::AR_REPOSITORY no válido"; exit 1; }
          [[ "$CLOUD_RUN_SERVICE" =~ ^[a-z][a-z0-9-]{0,48}$ ]] || { echo "::error::CLOUD_RUN_SERVICE no válido"; exit 1; }

      - name: Nombres de imagen
        env:
          REPOSITORIO: ${{ github.repository }}
        run: |
          echo "IMAGEN_GHCR=ghcr.io/${REPOSITORIO,,}/api:${GITHUB_SHA}" >> "$GITHUB_ENV"
          REGISTRO="${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT_ID}/${AR_REPOSITORY}"
          echo "IMAGEN_AR=${REGISTRO}/${CLOUD_RUN_SERVICE}:${GITHUB_SHA}" >> "$GITHUB_ENV"

      # Único paso de autenticación: OIDC + Workload Identity Federation.
      - name: Autenticar en GCP (OIDC, sin llaves)
        uses: google-github-actions/auth@7c6bc770dae815cd3e89ee6cdf493a5fab2cc093 # v3.0.0
        with:
          project_id: ${{ vars.GCP_PROJECT_ID }}
          workload_identity_provider: ${{ vars.GCP_WIF_PROVIDER }}
          service_account: ${{ vars.GCP_DEPLOY_SA }}

      - name: Configurar gcloud
        uses: google-github-actions/setup-gcloud@aa5489c8933f4cc7a4f7d45035b3b1440c9c10db # v3.0.1

      - name: Promocionar la imagen de GHCR a Artifact Registry
        run: |
          gcloud auth configure-docker "${GCP_REGION}-docker.pkg.dev" --quiet
          if ! docker pull "${IMAGEN_GHCR}"; then
            echo "::error::No existe ${IMAGEN_GHCR}. Espera a que el CI de main la publique (y que el paquete sea público)."
            exit 1
          fi
          docker tag "${IMAGEN_GHCR}" "${IMAGEN_AR}"
          docker push "${IMAGEN_AR}"
          if [[ "${GITHUB_REF}" == refs/tags/v* ]]; then
            # Etiqueta de versión adicional; la de SHA sigue siendo la que se despliega.
            ETIQUETA="${IMAGEN_AR%:*}:${GITHUB_REF_NAME}"
            docker tag "${IMAGEN_GHCR}" "${ETIQUETA}"
            docker push "${ETIQUETA}"
          fi

      - name: Desplegar en Cloud Run
        id: desplegar
        run: |
          # Sin --allow-unauthenticated: la cuenta de despliegue no puede cambiar
          # políticas IAM (privilegio mínimo). El acceso público se concede una vez,
          # a mano, en el primer despliegue (api/deploy/gcp/desplegar_manual.sh).
          gcloud run deploy "${CLOUD_RUN_SERVICE}" \
            --project "${GCP_PROJECT_ID}" \
            --region "${GCP_REGION}" \
            --image "${IMAGEN_AR}" \
            --service-account "${GCP_RUNTIME_SA}" \
            --port 8080 \
            --max-instances 1 \
            --set-env-vars "APP_VERSION=${GITHUB_SHA}" \
            --quiet
          URL="$(gcloud run services describe "${CLOUD_RUN_SERVICE}" \
            --project "${GCP_PROJECT_ID}" --region "${GCP_REGION}" \
            --format='value(status.url)')"
          echo "url=${URL}" >> "$GITHUB_OUTPUT"

      - name: Comprobar /health y la versión desplegada
        env:
          URL: ${{ steps.desplegar.outputs.url }}
        run: |
          for intento in 1 2 3 4 5 6; do
            if cuerpo="$(curl -fsS --max-time 10 "${URL}/health")"; then
              echo "${cuerpo}"
              grep -q "\"version\":\"${GITHUB_SHA}\"" <<< "${cuerpo}" \
                || { echo "::error::La versión desplegada no es ${GITHUB_SHA}"; exit 1; }
              exit 0
            fi
            echo "Intento ${intento}: aún no responde; reintento en 10 s"
            sleep 10
          done
          echo "::error::${URL}/health no respondió"; exit 1
```

El paso esencial de autenticación usa ```workload_identity_provider``` y ```service_account```, no una llave. A continuación el *job* descarga la imagen de *GHCR*, la etiqueta y la publica en *Artifact Registry*, despliega en *Cloud Run* con la cuenta de ejecución y comprueba que ```GET /health``` responde y que ```version``` es el *SHA* del *commit*.

Si la imagen aún no está en *GHCR*, el paso de promoción falla con un mensaje que pide esperar al *CI* de ```main``` y a que el paquete sea público.

## Actividad.

1. Definir en la sesión ```GCP_PROJECT_ID``` y ```GCP_REGION```, ejecutar ```preparar_proyecto.sh``` y ```configurar_wif.sh```, y cargar en el entorno ```cloudrun``` las variables ```GCP_PROJECT_ID```, ```GCP_REGION```, ```AR_REPOSITORY```, ```CLOUD_RUN_SERVICE```, ```GCP_WIF_PROVIDER```, ```GCP_DEPLOY_SA``` y ```GCP_RUNTIME_SA```.
2. Ejecutar el primer despliegue manual y abrir la *URL* del servicio, comprobando ```GET /health```.
3. Disparar el *workflow* ```Despliegue en Cloud Run``` (después de un *CI* verde en ```main```) y verificar que la revisión usa la imagen promocionada desde *GHCR* con el *SHA* del *commit*, y que ```APP_VERSION``` en ```/health``` coincide con ese *SHA*.
4. Señalar, en el *workflow*, el paso de *WIF*, el de promoción (sin reconstruir) y la ausencia de ```--allow-unauthenticated```.

## Mapa breve de servicios que no se practican aquí.

* **Cloud SQL.** Base de datos relacional gestionada. En cursos posteriores se conecta a la aplicación (a menudo mediante un *proxy* de autenticación, sin exponer el puerto de la base a internet).
* **Secret Manager.** Almacén de secretos de *GCP*. En cursos posteriores la aplicación lee de ahí contraseñas o llaves, en lugar de llevarlas en la imagen o en el repositorio.
* **Compute Engine.** Máquinas virtuales. El cuaderno 17 practica un servidor *Linux* como destino; no se aprovisiona esa máquina con *Compute Engine* en este curso.
* **Terraform.** Herramienta de infraestructura como código. Aquí los recursos se crean con los guiones de ```gcloud``` para entender el destino; en cursos posteriores esos mismos destinos se declaran como código.

Saber que existen basta para este curso. No hace falta configurarlos en este laboratorio.

## Limpieza.

Al terminar, ```api/deploy/gcp/limpiar.sh``` borra el servicio, el repositorio de *Artifact Registry*, el fondo de *WIF* y las cuentas de servicio de este curso, para no dejar costos ni accesos abiertos. Pide confirmación. También conviene borrar las variables del entorno ```cloudrun``` en *GitHub*.

La siguiente celda ejecutará la limpieza de los recursos de este laboratorio en el proyecto.

In [ ]:
bash api/deploy/gcp/limpiar.sh

## Cierre.

El estudiante ya configuró *WIF*, publicó o promocionó una imagen en *Artifact Registry* y la sirvió en *Cloud Run* con un *workflow* sin llaves. El cuaderno 17 hace el ejercicio equivalente sobre un *Ubuntu* 24.04 completo, alcanzado por *SSH*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>